# 5.3 Orthonormal Bases: The Gram–Schmidt Process

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 5:** Inner Product Spaces

---

### What you will learn

- How to **project** one vector onto another using the orthogonal projection formula.
- How the **Gram–Schmidt process** converts any linearly independent set into an **orthonormal** set.
- Why orthonormal bases make coordinate computation trivial via **Fourier coefficients**.
- How Gram–Schmidt connects to the **QR factorization** used throughout numerical linear algebra.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.inner import (
    dot, norm, normalize, are_orthogonal,
    orthogonal_projection, gram_schmidt,
)
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

---

## 2. Motivation

Given a non-orthogonal basis for a subspace, how do we build an **orthogonal** one
that spans the same space?

The **Gram–Schmidt process** answers this question. It is one of the most
important algorithms in linear algebra because:

1. **Coordinates become trivial.** In an orthonormal basis $Q = \{\mathbf{q}_1, \ldots, \mathbf{q}_n\}$,
   the coordinate of any vector $\mathbf{x}$ along $\mathbf{q}_i$ is simply the
   inner product $\langle \mathbf{x}, \mathbf{q}_i \rangle$ — no system-solving needed.

2. **QR factorization.** Every matrix $A$ with linearly independent columns can be
   written as $A = QR$, where $Q$ is orthogonal and $R$ is upper-triangular.
   Gram–Schmidt is the algorithm behind this factorization.

3. **Numerical stability.** Orthonormal bases avoid the ill-conditioning that
   plagues nearly-dependent bases.

The core idea is beautifully simple: **subtract off projections** onto the
directions you have already orthogonalized.

---

## 3. Build — Core Concepts

We build up the Gram–Schmidt process step by step, starting from the
orthogonal projection and ending with Fourier coefficients.

### 3.1 Orthogonal Projection

> **Definition (Larson, Sec. 5.3).** The **orthogonal projection** of $\mathbf{v}$
> onto $\mathbf{u}$ is
>
> $$\text{proj}_{\mathbf{u}}(\mathbf{v}) = \frac{\langle \mathbf{v}, \mathbf{u} \rangle}{\langle \mathbf{u}, \mathbf{u} \rangle} \, \mathbf{u}$$
>
> The **component of $\mathbf{v}$ orthogonal to $\mathbf{u}$** is
>
> $$\mathbf{v} - \text{proj}_{\mathbf{u}}(\mathbf{v})$$

Geometrically, $\text{proj}_{\mathbf{u}}(\mathbf{v})$ is the shadow of $\mathbf{v}$ onto
the line spanned by $\mathbf{u}$. The remainder $\mathbf{v} - \text{proj}_{\mathbf{u}}(\mathbf{v})$
is perpendicular to $\mathbf{u}$.

In [ ]:
u = [2, 1]
v = [3, 4]

proj = orthogonal_projection(v, u)
perp = [v[i] - proj[i] for i in range(len(v))]

print(f"u          = {u}")
print(f"v          = {v}")
print(f"proj_u(v)  = {proj}")
print(f"v - proj   = {perp}")
print()
print(f"dot(u, v - proj) = {dot(u, perp):.10f}  (should be ~0)")
print(f"Orthogonal?  {are_orthogonal(u, perp)}")

### 3.2 Gram–Schmidt Step 1 — $\mathbf{v}_1 = \mathbf{w}_1$

Given linearly independent vectors $\{\mathbf{w}_1, \mathbf{w}_2, \mathbf{w}_3\}$,
the Gram–Schmidt process builds an orthogonal set $\{\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3\}$.

**Step 1:** The first vector is kept unchanged:

$$\mathbf{v}_1 = \mathbf{w}_1$$

There is nothing to orthogonalize against yet.

In [ ]:
w1 = [1, 1, 0]
w2 = [1, 0, 1]
w3 = [0, 1, 1]

v1 = list(w1)

print(f"w1 = {w1}")
print(f"v1 = {v1}  (unchanged)")
print(f"||v1|| = {norm(v1):.6f}")

### 3.3 Gram–Schmidt Step 2 — $\mathbf{v}_2 = \mathbf{w}_2 - \text{proj}_{\mathbf{v}_1}(\mathbf{w}_2)$

**Step 2:** Subtract the component of $\mathbf{w}_2$ along $\mathbf{v}_1$:

$$\mathbf{v}_2 = \mathbf{w}_2 - \text{proj}_{\mathbf{v}_1}(\mathbf{w}_2) = \mathbf{w}_2 - \frac{\langle \mathbf{w}_2, \mathbf{v}_1 \rangle}{\langle \mathbf{v}_1, \mathbf{v}_1 \rangle} \, \mathbf{v}_1$$

After this subtraction, $\mathbf{v}_2 \perp \mathbf{v}_1$.

In [ ]:
proj_v1_w2 = orthogonal_projection(w2, v1)
v2 = [w2[i] - proj_v1_w2[i] for i in range(len(w2))]

print(f"w2                    = {w2}")
print(f"proj_v1(w2)           = {proj_v1_w2}")
print(f"v2 = w2 - proj_v1(w2) = {v2}")
print()
print(f"dot(v1, v2) = {dot(v1, v2):.10f}  (should be ~0)")
print(f"v1 \u22a5 v2?   {are_orthogonal(v1, v2)}")

### 3.4 Gram–Schmidt Step 3 — $\mathbf{v}_3 = \mathbf{w}_3 - \text{proj}_{\mathbf{v}_1}(\mathbf{w}_3) - \text{proj}_{\mathbf{v}_2}(\mathbf{w}_3)$

**Step 3:** Subtract projections onto **both** previously found orthogonal vectors:

$$\mathbf{v}_3 = \mathbf{w}_3 - \text{proj}_{\mathbf{v}_1}(\mathbf{w}_3) - \text{proj}_{\mathbf{v}_2}(\mathbf{w}_3)$$

This ensures $\mathbf{v}_3$ is orthogonal to **both** $\mathbf{v}_1$ and $\mathbf{v}_2$.

In [ ]:
proj_v1_w3 = orthogonal_projection(w3, v1)
proj_v2_w3 = orthogonal_projection(w3, v2)
v3 = [w3[i] - proj_v1_w3[i] - proj_v2_w3[i] for i in range(len(w3))]

print(f"w3                              = {w3}")
print(f"proj_v1(w3)                     = {proj_v1_w3}")
print(f"proj_v2(w3)                     = {proj_v2_w3}")
print(f"v3 = w3 - proj_v1(w3) - proj_v2(w3) = [{v3[0]:.6f}, {v3[1]:.6f}, {v3[2]:.6f}]")
print()
print(f"dot(v1, v3) = {dot(v1, v3):.10f}  (should be ~0)")
print(f"dot(v2, v3) = {dot(v2, v3):.10f}  (should be ~0)")
print(f"v1 \u22a5 v3?   {are_orthogonal(v1, v3)}")
print(f"v2 \u22a5 v3?   {are_orthogonal(v2, v3)}")

### 3.5 General Gram–Schmidt — Using `gram_schmidt`

> **Theorem (Larson, Sec. 5.3).** If $\{\mathbf{w}_1, \ldots, \mathbf{w}_n\}$ is a
> linearly independent set in an inner product space $V$, then the
> Gram–Schmidt process produces an **orthonormal** set
> $\{\mathbf{q}_1, \ldots, \mathbf{q}_n\}$ such that
>
> $$\text{span}(\mathbf{q}_1, \ldots, \mathbf{q}_k) = \text{span}(\mathbf{w}_1, \ldots, \mathbf{w}_k) \quad \text{for every } k = 1, \ldots, n$$

The `gram_schmidt` function performs both **orthogonalization** (subtract projections)
and **normalization** (divide by norm) in a single pass.

The general algorithm for $k = 1, \ldots, n$:

$$\mathbf{v}_k = \mathbf{w}_k - \sum_{j=1}^{k-1} \text{proj}_{\mathbf{v}_j}(\mathbf{w}_k), \qquad \mathbf{q}_k = \frac{\mathbf{v}_k}{\|\mathbf{v}_k\|}$$

In [ ]:
vectors = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]

Q = gram_schmidt(vectors)

print("Input vectors:")
for i, w in enumerate(vectors):
    print(f"  w{i+1} = {w}")

print("\nOrthonormal basis (Gram\u2013Schmidt):")
for i, q in enumerate(Q):
    print(f"  q{i+1} = [{q[0]:+.6f}, {q[1]:+.6f}, {q[2]:+.6f}]")

print("\nVerification:")
for i in range(len(Q)):
    print(f"  ||q{i+1}|| = {norm(Q[i]):.10f}")

print()
for i in range(len(Q)):
    for j in range(i + 1, len(Q)):
        d = dot(Q[i], Q[j])
        print(f"  dot(q{i+1}, q{j+1}) = {d:+.10f}")

### 3.6 Step-by-Step Projection Demo

To solidify understanding, let us trace **every intermediate computation**
of the Gram–Schmidt process: the projection being subtracted at each step
and the resulting orthogonal vector before normalization.

In [ ]:
print("=" * 60)
print("GRAM\u2013SCHMIDT: STEP-BY-STEP TRACE")
print("=" * 60)

ws = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]
orthogonal = []

for k, w in enumerate(ws):
    print(f"\n--- Step {k+1}: Processing w{k+1} = {w} ---")
    u = list(w)

    for j, vj in enumerate(orthogonal):
        proj = orthogonal_projection(u, vj)
        print(f"  proj_v{j+1}(w{k+1}) = [{proj[0]:+.6f}, {proj[1]:+.6f}, {proj[2]:+.6f}]")
        print(f"    \u27e8w{k+1}, v{j+1}\u27e9 = {dot(u, vj):.6f},  \u27e8v{j+1}, v{j+1}\u27e9 = {dot(vj, vj):.6f}")
        u = [u[i] - proj[i] for i in range(len(u))]
        print(f"    after subtraction: [{u[0]:+.6f}, {u[1]:+.6f}, {u[2]:+.6f}]")

    orthogonal.append(u)
    print(f"  v{k+1} (orthogonal) = [{u[0]:+.6f}, {u[1]:+.6f}, {u[2]:+.6f}]")
    print(f"  ||v{k+1}|| = {norm(u):.6f}")

    q = normalize(u)
    print(f"  q{k+1} (normalized) = [{q[0]:+.6f}, {q[1]:+.6f}, {q[2]:+.6f}]")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

### 3.7 Fourier Coefficients

> **Theorem (Larson, Sec. 5.3).** If $Q = \{\mathbf{q}_1, \ldots, \mathbf{q}_n\}$ is
> an **orthonormal** basis for a subspace $W$, then for any $\mathbf{x} \in W$,
>
> $$\mathbf{x} = \langle \mathbf{x}, \mathbf{q}_1 \rangle \, \mathbf{q}_1 + \langle \mathbf{x}, \mathbf{q}_2 \rangle \, \mathbf{q}_2 + \cdots + \langle \mathbf{x}, \mathbf{q}_n \rangle \, \mathbf{q}_n$$

The scalars $c_i = \langle \mathbf{x}, \mathbf{q}_i \rangle$ are called
**Fourier coefficients**. Compare this with a general basis, where finding
coordinates requires solving a linear system — here it is just $n$ dot products.

This is one of the main **practical rewards** of having an orthonormal basis.

In [ ]:
x = [5, 3, 7]
Q = gram_schmidt([[1, 1, 0], [1, 0, 1], [0, 1, 1]])

print(f"x = {x}")
print(f"Orthonormal basis Q:")
for i, q in enumerate(Q):
    print(f"  q{i+1} = [{q[0]:+.6f}, {q[1]:+.6f}, {q[2]:+.6f}]")

coeffs = [dot(x, q) for q in Q]
print(f"\nFourier coefficients:")
for i, c in enumerate(coeffs):
    print(f"  c{i+1} = \u27e8x, q{i+1}\u27e9 = {c:+.6f}")

reconstructed = [0.0] * len(x)
for c, q in zip(coeffs, Q):
    for i in range(len(reconstructed)):
        reconstructed[i] += c * q[i]

print(f"\nReconstructed x = [{reconstructed[0]:.6f}, {reconstructed[1]:.6f}, {reconstructed[2]:.6f}]")

error = math.sqrt(sum((x[i] - reconstructed[i])**2 for i in range(len(x))))
print(f"Reconstruction error = {error:.2e}  (should be ~0)")

---

## 4. Verify — Cross-Check with NumPy

We verify our from-scratch Gram–Schmidt against NumPy’s `np.linalg.qr`,
which computes the QR factorization $A = QR$. The columns of $Q$ should
span the same space as our orthonormal vectors (they may differ by sign).

In [ ]:
def check_pass(condition, label):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {label}")
    return condition

In [ ]:
vectors = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]
Q_ours = gram_schmidt(vectors)

print("--- Test 1: Orthonormality ---")

all_unit = all(abs(norm(q) - 1.0) < EPSILON for q in Q_ours)
check_pass(all_unit, "All vectors have unit norm")

all_orth = True
for i in range(len(Q_ours)):
    for j in range(i + 1, len(Q_ours)):
        d = abs(dot(Q_ours[i], Q_ours[j]))
        if d >= EPSILON:
            all_orth = False
check_pass(all_orth, "All pairs are orthogonal (dot ~0)")

print("\n--- Test 2: Same span as original vectors ---")

A = np.array(vectors, dtype=float).T
Q_np, R_np = np.linalg.qr(A)

Q_ours_np = np.array(Q_ours).T

P_ours = Q_ours_np @ Q_ours_np.T
P_numpy = Q_np @ Q_np.T

span_match = np.allclose(P_ours, P_numpy, atol=1e-9)
check_pass(span_match, "Projection matrices match (same column space)")

print("\n--- Test 3: Compare against np.linalg.qr ---")
print("Our Q (columns):")
print(Q_ours_np)
print("\nNumPy Q (columns):")
print(Q_np)
print("\n(Signs may differ \u2014 that is expected. The column spaces are identical.)")

for i in range(Q_np.shape[1]):
    q_us = Q_ours_np[:, i]
    q_them = Q_np[:, i]
    same_or_neg = np.allclose(q_us, q_them, atol=1e-9) or np.allclose(q_us, -q_them, atol=1e-9)
    check_pass(same_or_neg, f"Column {i}: matches NumPy (up to sign)")

In [ ]:
print("--- Test 4: Fourier coefficient reconstruction ---")

test_vectors = [
    [5, 3, 7],
    [1, 0, 0],
    [-2, 4, 1],
    [10, 10, 10],
]

Q_basis = gram_schmidt([[1, 1, 0], [1, 0, 1], [0, 1, 1]])

for x in test_vectors:
    coeffs = [dot(x, q) for q in Q_basis]
    recon = [sum(c * q[i] for c, q in zip(coeffs, Q_basis)) for i in range(len(x))]
    err = math.sqrt(sum((x[i] - recon[i])**2 for i in range(len(x))))
    check_pass(err < EPSILON, f"x={x}  reconstructed with error {err:.2e}")

print("\n--- Test 5: Gram\u2013Schmidt on a 4D set ---")

vecs_4d = [
    [1, 1, 1, 0],
    [0, 1, 1, 1],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
]

Q4 = gram_schmidt(vecs_4d)

all_unit_4d = all(abs(norm(q) - 1.0) < EPSILON for q in Q4)
check_pass(all_unit_4d, "4D: All vectors have unit norm")

all_orth_4d = True
for i in range(len(Q4)):
    for j in range(i + 1, len(Q4)):
        if abs(dot(Q4[i], Q4[j])) >= EPSILON:
            all_orth_4d = False
check_pass(all_orth_4d, "4D: All pairs are orthogonal")
check_pass(len(Q4) == 4, f"4D: Produced {len(Q4)} vectors (expected 4)")

---

## 5. Visualize — 3D Gram–Schmidt

We plot the original vectors $\{\mathbf{w}_1, \mathbf{w}_2, \mathbf{w}_3\}$ in **red**
and the orthonormalized vectors $\{\mathbf{q}_1, \mathbf{q}_2, \mathbf{q}_3\}$ in **blue**.
The orthonormal vectors form a crisp right-angled frame, while the originals
are tilted at various angles.

In [ ]:
original = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]
ortho = gram_schmidt(original)

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.set_title('Original Vectors (red) vs\nOrthonormal Basis (blue)', fontsize=11)

origin = [0, 0, 0]
colors_orig = ['#e74c3c', '#c0392b', '#a93226']
colors_orth = ['#3498db', '#2980b9', '#2471a3']

for i, w in enumerate(original):
    ax1.quiver(*origin, *w, color=colors_orig[i], arrow_length_ratio=0.08,
               linewidth=2.5, alpha=0.85)
    ax1.text(w[0]*1.1, w[1]*1.1, w[2]*1.1, f'w{i+1}',
             color=colors_orig[i], fontsize=10, fontweight='bold')

for i, q in enumerate(ortho):
    ax1.quiver(*origin, *q, color=colors_orth[i], arrow_length_ratio=0.1,
               linewidth=2.5, alpha=0.85)
    ax1.text(q[0]*1.15, q[1]*1.15, q[2]*1.15, f'q{i+1}',
             color=colors_orth[i], fontsize=10, fontweight='bold')

ax1.set_xlim([-0.5, 1.2])
ax1.set_ylim([-0.5, 1.2])
ax1.set_zlim([-0.5, 1.2])
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')
ax1.view_init(elev=25, azim=135)

ax2 = fig.add_subplot(1, 2, 2)
ax2.set_title('Pairwise Dot Products\n(Orthonormality Check)', fontsize=11)

n = len(ortho)
dot_matrix = np.array([[dot(ortho[i], ortho[j]) for j in range(n)] for i in range(n)])

im = ax2.imshow(dot_matrix, cmap='RdBu_r', vmin=-0.1, vmax=1.1)
ax2.set_xticks(range(n))
ax2.set_yticks(range(n))
ax2.set_xticklabels([f'q{i+1}' for i in range(n)])
ax2.set_yticklabels([f'q{i+1}' for i in range(n)])

for i in range(n):
    for j in range(n):
        ax2.text(j, i, f'{dot_matrix[i, j]:.4f}',
                 ha='center', va='center', fontsize=10,
                 color='white' if abs(dot_matrix[i, j]) > 0.5 else 'black')

plt.colorbar(im, ax=ax2, shrink=0.8)

fig.suptitle('Gram\u2013Schmidt Orthonormalization in $\\mathbb{R}^3$',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Apply the Gram–Schmidt process **by hand** (using `orthogonal_projection` and
`normalize`) to the basis $\{\mathbf{w}_1, \mathbf{w}_2\}$ where

$$\mathbf{w}_1 = \begin{bmatrix} 3 \\ 4 \end{bmatrix}, \quad \mathbf{w}_2 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$$

Verify that the resulting two vectors are orthonormal.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Apply `gram_schmidt` to the following three vectors in $\mathbb{R}^4$:

$$\mathbf{w}_1 = \begin{bmatrix} 1 \\ 1 \\ 0 \\ 0 \end{bmatrix}, \quad
\mathbf{w}_2 = \begin{bmatrix} 1 \\ 0 \\ 1 \\ 0 \end{bmatrix}, \quad
\mathbf{w}_3 = \begin{bmatrix} 0 \\ 0 \\ 1 \\ 1 \end{bmatrix}$$

Then verify:
1. All norms are 1.
2. All pairwise dot products are 0.
3. Use Fourier coefficients to express $\mathbf{x} = [2, 3, 1, 4]$ in the
   orthonormal basis, and reconstruct $\mathbf{x}$ from those coefficients.
   (Note: $\mathbf{x}$ may not lie in the span of three 4D vectors, so the
   reconstruction will be the **projection** of $\mathbf{x}$ onto that 3D subspace.)

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Implement `qr_factorization(columns)` that takes a list of column vectors
(each a list of floats) and returns two NumPy arrays $(Q, R)$ such that
$A = QR$, where:

- $Q$ is orthogonal ($Q^T Q = I$), with columns from Gram–Schmidt.
- $R$ is upper-triangular, with $r_{ij} = \langle \mathbf{a}_j, \mathbf{q}_i \rangle$.

**Hint:** The entry $r_{ij}$ is the dot product of the $j$-th original column
with the $i$-th orthonormal vector. Since $\mathbf{q}_i$ is orthonormal,
$r_{ij} = \langle \mathbf{a}_j, \mathbf{q}_i \rangle$.

Test on

$$A = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}$$

and verify that $QR = A$ (up to floating-point tolerance) and that
$Q^T Q \approx I$.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Orthogonal projection** | $\text{proj}_{\mathbf{u}}(\mathbf{v}) = \frac{\langle \mathbf{v}, \mathbf{u} \rangle}{\langle \mathbf{u}, \mathbf{u} \rangle} \mathbf{u}$ gives the component of $\mathbf{v}$ along $\mathbf{u}$ |
| **Gram–Schmidt process** | Subtract projections iteratively: $\mathbf{v}_k = \mathbf{w}_k - \sum_{j<k} \text{proj}_{\mathbf{v}_j}(\mathbf{w}_k)$, then normalize |
| **Orthonormal basis** | All vectors have unit norm and are mutually orthogonal |
| **Fourier coefficients** | In an orthonormal basis $Q$, coordinates of $\mathbf{x}$ are $c_i = \langle \mathbf{x}, \mathbf{q}_i \rangle$ — no system-solving needed |
| **QR factorization** | $A = QR$ where $Q$ comes from Gram–Schmidt and $R_{ij} = \langle \mathbf{a}_j, \mathbf{q}_i \rangle$ |
| **Same span** | $\text{span}(\mathbf{q}_1, \ldots, \mathbf{q}_k) = \text{span}(\mathbf{w}_1, \ldots, \mathbf{w}_k)$ at every stage |